In [0]:
from datetime import date, datetime 
from pyspark.sql.functions import col 
from delta.tables import DeltaTable

#User pass date 
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=date.today().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema 
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

In [0]:
#Read booking bronze layer table
booking=spark.read.table(f"{catalog}.bronze.booking_inc").filter(col("business_date")==datetime.strptime(arrival_date, "%Y-%m-%d").date())
current_dim=spark.read.table(f"{catalog}.{schema}.customer_dim").filter(col("is_current")==True).select("customer_sk", "customer_id")

book_enriched=booking.join(current_dim, on="customer_id", how="left")

book_enriched_sel=book_enriched.select("booking_type", "customer_id", "customer_sk", "business_date", "amount",
"discount", "quantity")

agg=book_enriched_sel.groupBy("booking_type", "customer_sk", "customer_id", "business_date")\
.agg(sum("amount").alias("total_amount"), sum("discount").alias("total_discount"), sum("quantity").alias("total_quantity"))

fact_full_name=f"{catalog}.{schema}.booking_fact" 

#Create schema 
spark.sql(f"create schema if not exists {catalog}.{schema}")

try:
	if not spark.catalog.tableExists(fact_full_name):
		agg.limit(0).write.format("delta").mode("overwrite").saveAsTable(fact_full_name)

	target_table=DeltaTable.forName(spark, fact_full_name)

	target_table.alias("t").merge(agg.alias("s"), "t.booking_type=s.booking_type and s.customer_sk <> t.customer_sk and s.business_date=t.business_date")\
	.whenMatchedUpdate(set={"total_amount":"s.total_amount", "total_quantity":"s.total_quantity", "customer_id": "s.customer_id"})\
	.whenNotMatchedInsertAll() 

	print("Fact build complete (daily gain, surrogate key)")

except Exception as e:
	print("Fail to build fact build: {str(e)}")
	raise